# Cuaderno X: <Nombre del Pipeline>

Este cuaderno implementa una etapa del pipeline de procesamiento de datos siguiendo una estructura común reutilizable para todos los notebooks del proyecto. Su objetivo es ejecutar una fase específica del flujo de trabajo manteniendo un contrato de datos, una preparación homogénea del entorno y una auditoría consistente de los resultados.

La lógica particular de esta etapa se concentra únicamente en la fase de procesamiento, mientras que el resto de la estructura permanece estandarizada para facilitar la reutilización, el mantenimiento y su futura integración dentro de una aplicación.

## Índice de Contenidos

### 0. Configuración del Entorno
- Cargar Módulos de Jupyter Notebook.

### 1. Importación y Configuración Librerías
- Importación de librerías y dependencias.
- Configuración de librerías

### 2. Preparación del Pipeline
- Configuración de rutas y estado del pipeline.
- Definición del contrato de datos.
- Importación del dataset.
- Validación y normalización del dataset.
- Preparación del dataset para el pipeline.


## 0 Recarga Automática de Módulos

Durante el desarrollo del notebook se habilita la extensión `autoreload` de Jupyter, permitiendo recargar automáticamente los módulos externos cada vez que su código es modificado. Esto evita reiniciar el kernel tras realizar cambios en las librerías propias del proyecto y agiliza el ciclo de desarrollo.

In [ ]:
# ==============================================================================
# ETAPA 0. RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS DE JUPYTER
# ==============================================================================
%load_ext autoreload
%autoreload 2

### 1.1 Configuración del Entorno e Importación de Librerías

En esta etapa se prepara el entorno de ejecución y se importan todas las dependencias necesarias para el notebook, organizadas según su función para facilitar su mantenimiento y localización.

En primer lugar, se define de forma opcional una configuración del entorno destinada a limitar el paralelismo interno de determinadas librerías numéricas. Esta configuración resulta útil durante procesos de entrenamiento o validación para evitar la sobreutilización de recursos cuando el paralelismo ya es gestionado por el propio pipeline.

Posteriormente, se importan las librerías del proyecto, las dependencias base de Python para el tratamiento de datos y las librerías de visualización utilizadas durante el análisis. Cada grupo de importaciones incorpora un manejo explícito de errores que facilita la detección temprana de problemas de configuración o dependencias ausentes.

Al finalizar la etapa se confirma que el entorno de ejecución ha sido inicializado correctamente y que todas las dependencias requeridas se encuentran disponibles para el resto del notebook.

In [ ]:
# ==============================================================================
# ETAPA 1.1. Importación de librerías necesarias
# ==============================================================================
# region Environment -----------------------------------------------------------
def conf_env():
    global os
    try: 
        import os

        # "Capa" el paralelismo de bajo nivel antes de importar numpy o sklearn por si se los modelos no hacen caso al poner un proceso
        os.environ["OMP_NUM_THREADS"] = "1"
        os.environ["MKL_NUM_THREADS"] = "1"
        os.environ["OPENBLAS_NUM_THREADS"] = "1"
        os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
        os.environ["NUMEXPR_NUM_THREADS"] = "1"

    except Exception:
        print(f"Ha ocurrido un error al ajustar el entorno")
        raise

# AJUSTAR VARIABLES DE ENTORNO
# conf_env()

# endregion Environment --------------------------------------------------------

# region ds-utils Library ------------------------------------------------------
try:   
    from ds_utils.eda import preview_utilities
    from ds_utils.eda import plot_utilities
    from ds_utils.eda.preview_config import Color, Alignment

    from ds_utils.preprocessing import preprocessing_utilities

    from ds_utils.ml import ml_utilities

    from ds_utils.base_classes.dataset_contract import DatasetContract

except Exception:
    print(
        "Ha ocurrido un error al importar ds-utils. "
        "Si no está instalada puedes hacerlo con todos los extras mediante: "
        "pip install ds-utils[eda,ml] @ git+https://github.com/juanjosggarcia/ds-utils.git@v0.1.0"
    )
    raise
# endregion ds-utils Library ---------------------------------------------------

# region Base Libraries --------------------------------------------------------
try:
    import os 
    from pathlib import Path
    from dataclasses import dataclass, field

    # from dotenv import load_dotenv # DESCOMENTAR en caso de usar la fase "1.2 Configuración de Librerías"
    
    import numpy as np
    import pandas as pd

except Exception:
    print(f"Ha ocurrido un error al importar las librerías base (datos, matemáticas)")
    raise
# endregion Base Libraries -----------------------------------------------------

# region Visualization Libraries -----------------------------------------------
try:
    # import matplotlib.pyplot as plt
    ...

except Exception:
    print(f"Ha ocurrido un error al importar las librerías de visualización")
    raise
# endregion Visualization Libraries --------------------------------------------

# region Machine Learning Libraries --------------------------------------------
try:
    # from sklearn.base import BaseEstimator
    ...
    
except Exception:
    print(f"Ha ocurrido un error al importar las librerías de Machine Learning")
    raise
# endregion Machine Learning Libraries -----------------------------------------

preview_utilities.print_console_section("Fase 1.1 Configuración del Entorno e Importaciones de Librerías".upper(), "Fase finalizada correctamente...")

FASE 1.1 CONFIGURACIÓN DEL ENTORNO E IMPORTACIONES DE LIBRERÍAS
Fase finalizada correctamente...


### 1.2 Configuración de Librerías

Esta etapa es **opcional** y solo debe ejecutarse cuando el proyecto requiera configuración adicional de servicios o librerías externas.

Por ejemplo, un pipeline de Machine Learning puede necesitar:

* Seguridad: Carga de credenciales sensibles desde el archivo .env.
* Tracking (MLflow): Conexión al servidor para el registro de experimentos y métricas.
* Artefactos (MinIO/S3): Configuración del almacenamiento físico para guardar los modelos finales.
* Validación: Verificación en tiempo real de la conectividad antes de proceder.

Esta sección no es necesaria para pipelines centrados únicamente en limpieza, validación o EDA.

In [ ]:
# ==============================================================================
# ETAPA 1.2. CONFIGURACIÓN DE LIBRERÍAS
# ==============================================================================

SOLO EJECUTAR EN CASO DE USAR NECESITAR CONFIGURACIONES ADICIONALES CON LAS LIBRERIAS

# region Environment -----------------------------------------------------------
def conf_environment():
    """Configuración global de Entorno."""
    
    # Cargamos la configuracion de MLFlow desde las variables de entorno
    ENV_FILE = ".env"
    dotenv_path = Path.cwd().parents[2] / "secrets" / ENV_FILE

    if not dotenv_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo de entorno: '{dotenv_path}'")

    load_dotenv(dotenv_path=dotenv_path)
    
conf_environment()

# endregion Environment ---------------------------------------------------------

# region MLflow ----------------------------------------------------------------
def conf_mlflow():
    """Configuración global de MLflow."""
    
    key = "MLFLOW_ENDPOINT"
    value_mlflow_endpoint = os.getenv(key, "")
    if not value_mlflow_endpoint:
        raise FileNotFoundError(f"ENV key {key} not found.")
    
    mlflow.set_tracking_uri(value_mlflow_endpoint)

    # Verificar la conexión con MLFlow
    try:
        experiments = mlflow.search_experiments()
        print("Conexión exitosa a MLflow")
    except Exception as e:
        print("Error conectando a MLflow:", e)
        raise e
    
conf_mlflow()

# endregion MLflow -------------------------------------------------------------


# region MinIO/S3 --------------------------------------------------------------
def conf_minio():
    """Configuración global del cliente MinIO."""
    
    key = "MINIO_USER"
    value_minio_user = os.getenv(key, "")
    if not value_minio_user:
        raise FileNotFoundError(f"ENV key {key} not found.")


    os.environ["AWS_ACCESS_KEY_ID"] = value_minio_user

    key = "MINIO_PASS"
    value_minio_pass = os.getenv(key, "")
    if not value_minio_pass:
        raise FileNotFoundError(f"ENV key {key} not found.")
    
    os.environ["AWS_SECRET_ACCESS_KEY"] = value_minio_pass

    key = "MINIO_ENDPOINT"
    value_minio_endpoint = os.getenv(key, "")
    if not value_minio_endpoint:
        raise FileNotFoundError(f"ENV key {key} not found.")
    
    # uri de la api de minio
    os.environ["MLFLOW_S3_ENDPOINT_URL"] = value_minio_endpoint

    # CONEXIÓN CON MINIO / S3

    boto3_session = boto3.Session()

    # Configura el cliente S3
    s3_client = boto3_session.client('s3', endpoint_url=value_minio_endpoint)

    # Verificar la conexión al bucket
    try:
        # s3_client.list_objects(Bucket='mlflow')
        buckets = s3_client.list_buckets()
        print("Conexión exitosa a S3/MinIO")
    except NoCredentialsError as e:
        print("Error conectando a S3/MinIO. Posibles Credenciales incorrectas o no configuradas")
        raise e

conf_minio()

# endregion MinIO/S3 -----------------------------------------------------------

preview_utilities.print_console_section("Fase 1.2 Configuración de Librerías".upper(), "Fase finalizada correctamente...")

## 2. Preparación del Dataset

En esta etapa se realiza la incorporación del conjunto de datos al pipeline de trabajo y se verifica que cumple la estructura definida por el contrato de datos. Durante este proceso se llevan a cabo las validaciones comunes, la normalización necesaria para unificar el formato de entrada y la preparación del conjunto de datos para su utilización en las siguientes etapas del análisis.

El objetivo es obtener un DataFrame consistente, auditado y preparado para ser procesado, manteniendo en todo momento la trazabilidad entre el conjunto de datos original y la versión normalizada que utilizará el pipeline.

### 2.1 Configuración de Rutas y Estado del Pipeline

En esta etapa se definen las rutas de entrada y salida utilizadas por el notebook para la carga y persistencia de los datos procesados. Centralizar esta configuración facilita la reutilización del cuaderno y evita la dispersión de rutas a lo largo del código.

Asimismo, se inicializa un objeto `PipelineState`, encargado de registrar las modificaciones estructurales realizadas sobre el DataFrame durante la ejecución del pipeline. En concreto, mantiene un seguimiento de las columnas creadas y eliminadas, proporcionando una base para la auditoría y el resumen final del procesamiento.

La información almacenada en este objeto es independiente de la lógica específica de cada notebook, permitiendo reutilizar el mismo mecanismo de seguimiento en cualquier etapa del proyecto.

In [ ]:
# ==============================================================================
# ETAPA 2.1. CONFIGURACIÓN DE RUTAS Y PIPELINE DEL NOTEBOOK
# ==============================================================================

# Archivos externos de entrada de datos
DATA_FILE_NAME = "<dataset_name>.csv"
DATA_PATH = Path.cwd().parents[1] / "data" / DATA_FILE_NAME
# Archivos externos de salida de datos
ML_FLOW_EXPERIMENT_NAME = "<experiment_name>" # SOLO NECESARIO EN CASO DE ML
ML_FLOW_DATASET_BASE_NAME = "<dataset_name>" # SOLO NECESARIO EN CASO DE ML


@dataclass
class PipelineState:
    """Registra el estado del pipeline durante su ejecución."""

    added_columns: set[str] = field(default_factory=set)
    """Columnas generadas y añadidas al DataFrame durante el procesamiento."""

    removed_columns: set[str] = field(default_factory=set)
    """Columnas eliminadas del DataFrame durante el procesamiento."""


# Instancia global del pipeline para su uso en el cuaderno
pipeline = PipelineState()


preview_utilities.print_console_section("Fase 2.1 Configuración de Rutas y Definición del PipelineState del Notebook".upper(), "Fase finalizada correctamente...")

### 2.2 Definición del Contrato de Datos

En esta etapa se define el contrato de datos asociado al dataset que será procesado durante el pipeline. Este contrato centraliza la nomenclatura de columnas, agrupaciones de datos y parámetros de configuración, evitando dependencias directas con los nombres originales proporcionados por la fuente de datos.

La clase `DatasetContract` actúa como una capa de abstracción entre los datos externos y las diferentes etapas de procesamiento. De esta forma, cualquier cambio en la estructura del dataset original puede gestionarse modificando únicamente el contrato, manteniendo desacoplada la lógica del pipeline.

Dentro del contrato se definen diferentes grupos de información:

- **Columnas base del dataset:** contienen los nombres canónicos de las columnas procedentes de la fuente original.
- **Columnas de características:** reservan la nomenclatura de variables generadas durante fases posteriores de transformación o enriquecimiento.
- **Grupos de columnas:** permiten identificar conjuntos de variables con una finalidad común, como columnas requeridas, columnas utilizadas durante el procesamiento o columnas originales del dataset.
- **Columnas de normalización:** registran las modificaciones estructurales realizadas durante la adaptación inicial del dataset, como columnas creadas o eliminadas durante la generación de una representación temporal común.
- **Reglas y parámetros asociados:** almacenan configuraciones específicas necesarias para aplicar operaciones posteriores de transformación o validación.

El contrato se instancia como un objeto global para que todas las etapas del notebook puedan utilizar la misma referencia, garantizando consistencia en nombres y configuración a lo largo de todo el flujo de procesamiento.

In [ ]:
# ==============================================================================
# ETAPA 2.2. CONFIGURACIÓN DEL CONTRATO DE DATOS
# ==============================================================================

@dataclass(frozen=True)
class MetricsContract(DatasetContract):

    # region BASE DATASET COLUMNS ----------------------------------------------

    # --- Timestamp ---
    COL_TIME = "timestamp" # Canonical name for the contract (It must be created if not present in the source dataset.)

     # --- RAW Columns ---
    COL_CPU = "cpu"
    COL_RAM = "ram"
    COL_SESSIONS = "sessions"

    # endregion BASE DATASET COLUMNS -------------------------------------------

    # region FEATURE COLUMNS ---------------------------------------------------

    # --- Temporary Features ---
    COL_HOUR = 'hour'
    COL_HOUR_SIN = 'hour_sin'
    COL_HOUR_COS = 'hour_cos'

    COL_WEEK_DAY = 'weekday'
    COL_WEEK_SIN = 'week_sin'
    COL_WEEK_COS = 'week_cos'

    COL_WEEK_DAY_TEXT = 'weekday_text'

    COL_BUSINESS_HOUR = "is_business_hours"
    COL_WEEKEND_DAY = "is_weekend"

    # --- Clean objetive data ---
    COL_CPU_CLEAN = f'{COL_CPU}_clean'
    COL_RAM_CLEAN = f'{COL_RAM}_clean'

    # endregion FEATURE COLUMNS ------------------------------------------------

    # region COLUMN GROUPS -----------------------------------------------------

    # --- Original data columns list ---
    DATA_COLUMNS = [
        COL_CPU,
        COL_RAM,
        COL_SESSIONS
    ]
    '''Core data columns from the source dataset.'''

    # --- Required columns list ---
    REQUIRED_COLUMNS = [
        COL_TIME,
        *DATA_COLUMNS,
    ]
    '''Columns that must exist in the input dataset. Their existence must be validated.'''

    # --- Used columns list ---
    USED_COLUMNS = [
        COL_CPU,
        COL_RAM,
        COL_SESSIONS
    ]
    '''Columns processed throughout the pipeline.'''

    # endregion COLUMN GROUPS --------------------------------------------------

    # region NORMALIZATION -----------------------------------------------------
   
    # --- Normalization removed columns ---
    NORMALIZATION_REMOVED_COLUMNS  = []
    '''Columns discarded during the normalization stage.'''
   
    # --- Normalization created columns ---
    NORMALIZATION_CREATED_COLUMNS = []
    '''Columns created during the normalization stage.'''

    # endregion NORMALIZATION --------------------------------------------------

    # region DATA CLEANING -----------------------------------------------------

    # si no es porcentaje usar el P99 o el IQR (Interquartile Range) q3 - q1
    # --- Tolerances dict ---
    TOLERANCES = {
        COL_CPU: 50.0,
        COL_RAM: 30.0,
        COL_SESSIONS: None
    }
    '''Dictionary of cleaning thresholds applied to each processing column.'''

    # endregion DATA CLEANING --------------------------------------------------

    # region DATA ENGINEERING --------------------------------------------------

    SAMPLING_FREQUENCY_MIN: int = 5
    FREQ_RECORDS_PANDAS_OFFSET: str = "5min" # Pandas offset alias (e.g., "30s", "5min", "1h", "1D", "1W")

    CLEANING_WINDOW: int = 3

    LOOKBACK_STEPS: int = 6

    WINDOW_ROLLING: int = 5

    # endregion DATA ENGINEERING -----------------------------------------------

    # region MACHINE LEARNING --------------------------------------------------

    FORECAST_HORIZON: int = 12

    TEST_SIZE: float = 0.15

    VALIDATION_SIZE: float = 0.18 # 18% del restante al quitar la parte de Test


    RANDOM_STATE: int = 42

    N_TIMESERIES_SPLITS: int = 3

    EARLY_STOPPING_ROUNDS: int = 50
    
    # endregion MACHINE LEARNING -----------------------------------------------

    # region CONTRACT VALIDATIONS ----------------------------------------------

    # EJEMPLO DE IMPLEMENTACIÓN, ELIMINAR EN CASO DE NO NECESITARLO
    def validate_contract(self):
            
        # Contract Validation Pipeline
        self._validate_used_tolerances_columns()

    def _validate_used_tolerances_columns(self):
        ...

    # endregion CONTRACT VALIDATIONS -------------------------------------------

    # region DATASET VALIDATIONS -----------------------------------------------

    # EJEMPLO DE IMPLEMENTACIÓN, ELIMINAR EN CASO DE NO NECESITARLO
    def normalize_dataset(self, df):
        
        # Dataset Normalization Pipeline
        return self._get_canonical_time(df)
    
    def _get_canonical_time(self, df):
       ...

    # endregion DATASET VALIDATIONS --------------------------------------------

# Instancia global del contrato para su uso en el cuaderno
contract = MetricsContract()

# ==============================================================================
# COLUMNA OBJETIVO PARA PREDECIR
# ==============================================================================
COL_TARGET = contract.COL_CPU_CLEAN # EJEMPLO DE IMPLEMENTACIÓN, ELIMINAR EN CASO DE NO NECESITARLO


preview_utilities.print_console_section("Fase 2.2 Definición del Contrato de Datos".upper(), "Fase finalizada correctamente...")

### 2.3 Carga del Dataset

En esta etapa se carga el conjunto de datos desde el archivo de entrada definido en la configuración del notebook.

Antes de realizar la lectura, se comprueba que el archivo exista en la ruta especificada para evitar errores durante la ejecución. Una vez importado, el dataset se almacena en `df_raw`, que representa la versión original de los datos y se conserva sin modificaciones durante todo el pipeline.

Finalmente, se muestra una previsualización de las primeras filas del dataset para verificar que la importación se ha realizado correctamente y disponer de una primera inspección de la información contenida en el archivo.

In [ ]:
# ==============================================================================
# ETAPA 2.3. IMPORTACIÓN DEL DATASET
# ==============================================================================

try:
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"No se encuentra el archivo en: {DATA_PATH}")

    # Lectura del DataFrame de la memoria
    df_raw = pd.read_csv(DATA_PATH)
    
except FileNotFoundError as e:
    print(f"Error de archivo: {e}")
    raise e
except Exception as e:
    print(f"Ha ocurrido un error inesperado: {e}")
    raise e

# ==============================================================================  
# VISUALIZACIÓN DE RESULTADOS
# ==============================================================================
# Previsualización del DataFrame original
preview_utilities.preview_dataframe(
    title = f"Primeras filas del Dataset RAW '{DATA_FILE_NAME}'. Total: filas{df_raw.shape}columnas", 
    df = df_raw, 
    header_align = Alignment.LEFT
)

### 2.4 Validación y Normalización del Dataset

Una vez importado el conjunto de datos, se aplica el contrato de datos definido para el cuaderno. El contrato constituye el punto de entrada del pipeline y garantiza que el resto del procesamiento trabaje sobre un esquema de datos consistente y controlado.

Durante esta etapa se realizan varias operaciones principales:

- **Validación del dataset**, comprobando que el conjunto de datos cumple los requisitos mínimos establecidos por el contrato.
- **Normalización del dataset**, aplicando las transformaciones necesarias para obtener el esquema canónico que será utilizado durante el resto del pipeline.
- **Copia del dataset original**, los cambios realizados se hacen sobre una copia del Dataset original para no alterar el mismo.

A continuación, se completan aquellos parámetros del contrato cuya configuración depende de las características estadísticas del propio conjunto de datos. Este enfoque permite mantener un contrato reutilizable y, al mismo tiempo, adaptar determinados parámetros a cada dataset de forma automática.

Por último, se muestran las diferencias entre el dataset original (`df_raw`) y el dataset resultante (`df`), así como un resumen de su estructura y distribución temporal, permitiendo verificar que la aplicación del contrato se ha realizado correctamente antes de continuar con las siguientes etapas del procesamiento.

In [ ]:
# ==============================================================================
# ETAPA 2.4. VALIDACIÓN Y NORMALIZACIÓN DEL DATASET
# ==============================================================================

try:
    # Valida y normaliza el conjunto de datos de acuerdo con el contrato definido. Se lanza error en caso que no se cumpla
    # Crea una copia con el dataset modificado para no alterar el original
    df = contract.apply(df_raw)

except KeyError as e:
    print(f"Error de nombres de columna: {e}")
    raise e
except Exception as e:
    print(f"Ha ocurrido un error inesperado: {e}")
    raise e

# ============================================================================== 
# CONFIGURACIÓN DINÁMICA DEL CONTRATO
# ============================================================================== 
# Calculamoa los umbrales estadísticos que dependen del conjunto de datos.
contract.TOLERANCES[contract.COL_SESSIONS] = df_raw[contract.COL_SESSIONS].quantile(0.99)


# ==============================================================================  
# VISUALIZACIÓN DE RESULTADOS
# ==============================================================================

preview_utilities.show_df_differences(df_raw, df)
preview_utilities.show_df_details(
    df, 
    col_time=contract.COL_TIME,
    full_info=True,
    resume_info=True,
    freq_missing=contract.FREQ_RECORDS_PANDAS_OFFSET,
    missing_gap_limit=6 # 30 minutos (6 pasos de 5 min)
)
preview_utilities.print_console_section("Fase 2.4. Validación y Normalización del Dataset".upper(), "Fase finalizada correctamente...")

### 2.5 Preparación del Dataset para el Pipeline

Una vez validado y normalizado el dataset, se realizan las adaptaciones necesarias para facilitar su procesamiento con Pandas durante el resto del cuaderno.

En esta etapa se convierte la columna temporal al tipo `datetime` de Pandas y se ordena el dataset cronológicamente para garantizar un procesamiento temporal consistente. Estas operaciones no modifican el esquema definido por el contrato, sino que preparan el conjunto de datos para las etapas posteriores de análisis, limpieza e ingeniería de características.

Finalmente, se muestra una previsualización del dataset resultante (`df`), que será el utilizado durante el resto del pipeline.

In [ ]:
# ==============================================================================
# ETAPA 2.5 PREPARACIÓN DE DATOS PARA EL PIPELINE
# ==============================================================================

# 1. Normalizamos la columna de tiempo para pandas y creamos una copia del dataframe original sin añadir features extra
df = preprocessing_utilities.prepare_time_features(df, contract.COL_TIME)

# 2. Ordenamos de forma explicita el dataset. Se resetea el indice y usamos drop=True para evitar que el indice antiguo se guarde en una columna nueva
df = df.sort_values(contract.COL_TIME).reset_index(drop=True)

# 3. Fijar la columna de tiempo como el índice temporal (MEJOR PONERLO CUANDO SE NECESITE Y RESETEARLO AL TERMINAR)
# Sirve para: Resampling y agrupación temporal fácil, Selección rápida por fecha, Rolling por tiempo, etc.
# `inplace = true` modifica el dataframe original, no es buena practica porque puede influir en dataframe de celdas inferiores
# df = preprocessing_utilities.set_index_safe(df, contract.COL_TIME)

# ==============================================================================  
# VISUALIZACIÓN DE RESULTADOS
# ==============================================================================
# Previsualización de DataFrame para trabajar
preview_utilities.preview_dataframe(
    title = f"Primeras filas del Dataset Procesado '{DATA_FILE_NAME}'. Total: filas{df.shape}columnas", 
    df = df, 
    header_align = Alignment.LEFT
)